In [ ]:
pip install keras

In [ ]:
from tensorflow import keras
import numpy as np
from PIL import Image
import gradio as gr

modelo = keras.models.load_model("numeros_conv_ad_do.h5")


In [ ]:
import numpy as np
from PIL import Image #convertir imagen antes de mandarla

def predecir_digito(entrada):
    try:
        if entrada is None:
            return "No se recibió imagen"
        img = None

        #convierto la imagen
        if isinstance(entrada, dict):
            # revisar posibles claves
            for clave in ("image", "composite", "background"):
                if clave in entrada and entrada[clave] is not None:
                    dato = entrada[clave]
                    # puede venir como np.array o PIL
                    if isinstance(dato, np.ndarray):
                        dato = dato.astype("uint8")
                        img = Image.fromarray(dato)
                    else:
                        img = dato
                    break

            if img is None:
                return "No pude extraer la imagen del editor"

        else:
            # viene directo, no en dict
            if isinstance(entrada, np.ndarray):
                entrada = entrada.astype("uint8")
                img = Image.fromarray(entrada)
            else:
                img = entrada  # asumimos PIL.Image

        #extra
        if img is None:
            return "Formato de imagen no reconocido"

        #Pasar a escala de grises
        img = img.convert("L")

        #Redimensionar a 28x28 (como MNIST)
        img = img.resize((28, 28))

        # A numpy + normalizar
        arr = np.array(img).astype("float32") / 255.0

        #Invertir colores para que se parezca a MNIST
        arr = 1.0 - arr

        # Dar forma (1, 28, 28, 1)
        arr = arr.reshape(1, 28, 28, 1)

        #Predicción
        pred = modelo.predict(arr) #genra probabilidades
        digito = int(np.argmax(pred)) #la mas acertada

        return f"Predicción: {digito}"
    except Exception as e:
        return f"Ocurrió un error en la función: {repr(e)}"


In [ ]:
import gradio as gr


with gr.Blocks() as demo:
    gr.Markdown("# Reconocimiento de dígitos escritos a mano")
    gr.Markdown("Dibuja un número o sube una imagen y el modelo CNN (MNIST) intentará reconocerlo.")

    with gr.Tab("Dibujar"):
        lienzo = gr.ImageEditor(
            type="numpy",
            label="Dibuja aquí el número",
            height=280,
            width=280,
        )
        out_draw = gr.Textbox(label="Predicción")
        btn_draw = gr.Button("Reconocer dibujo")
        btn_draw.click(predecir_digito, inputs=lienzo, outputs=out_draw)

    with gr.Tab("Subir imagen"):
        img_up = gr.Image(
            type="numpy",
            image_mode="L",
            label="Sube una imagen de un dígito (0–9)"
        )
        out_up = gr.Textbox(label="Predicción")
        btn_up = gr.Button("Reconocer imagen")
        btn_up.click(predecir_digito, inputs=img_up, outputs=out_up)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step
